<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [25]:
import warnings

warnings.filterwarnings("ignore")

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install mlflow
import mlflow

In [27]:
mlflow.set_experiment(
    "assignment"
)

<Experiment: artifact_location='/content/mlruns/1', creation_time=1779037798646, experiment_id='1', last_update_time=1779037798646, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [28]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

def load_data(seed: int):
    fashion_mnist = fetch_openml(
        "Fashion-MNIST",
        version=1,
        as_frame=False
    )

    X = fashion_mnist.data.astype(np.float32)
    y = fashion_mnist.target.astype(np.int64)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    X_train = X_train / 255.0
    X_val = X_val / 255.0

    return X_train, X_val, y_train, y_val

A normalização é recomendada para modelos MLP, pois os pixels do Fashion MNIST variam de 0 a 255, e redes neurais tendem a treinar melhor quando as entradas estão em uma escala menor, como entre 0 e 1. Isso melhora a estabilidade numérica, favorece a convergência dos algoritmos de otimização e evita que valores muito altos prejudiquem o aprendizado. Além disso, normalizar os dados contribui para comparações mais justas entre experimentos registrados no MLflow. Neste caso, a normalização foi aplicada após a divisão dos dados, mantendo uma prática mais adequada para evitar vazamento de informação entre treino e validação.

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [23]:
from sklearn.neural_network import MLPClassifier

def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        solver="adam",
        max_iter=100,
        early_stopping=False
    )

    model.fit(X_train, y_train)

    return model

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    # Realiza predições
    y_pred = model.predict(X_test)

    # Calcula métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="macro")
    recall = recall_score(y_test, y_pred, average="macro")
    f1 = f1_score(y_test, y_pred, average="macro")

    # Retorna resultados
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [17]:
import time
import mlflow
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def run_experiment(
    X_train,
    X_val,
    y_train,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    max_iter,
    batch_size,
    seed
):
    with mlflow.start_run():
        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed
        )

        model.fit(X_train, y_train)

        training_time = time.time() - start_time

        y_pred = model.predict(X_val)

        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, average="macro")
        recall = recall_score(y_val, y_pred, average="macro")
        f1 = f1_score(y_val, y_pred, average="macro")

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("training_time", training_time)

        return model, {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "training_time": training_time
        }

# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [24]:
X_train, X_val, y_train, y_val = load_data(seed=42)

activations = ["logistic", "tanh", "relu"]

results = []

for activation in activations:
    model, metrics = run_experiment(
        X_train,
        X_val,
        y_train,
        y_val,
        activation=activation,
        hidden_layers=(100,),
        learning_rate=0.001,
        max_iter=100,
        batch_size=128,
        seed=42
    )

    results.append({
        "activation": activation,
        **metrics
    })

results_df = pd.DataFrame(results)
results_df

KeyboardInterrupt: 

## Responda:
- Qual ativação apresentou melhor convergência?
- Qual ativação apresentou maior estabilidade?
- Houve diferenças significativas de treinamento?

A função de ativação logistic apresentou a melhor convergência neste experimento, pois alcançou os maiores valores de accuracy, precision, recall e f1-score entre as três opções testadas, indicando desempenho ligeiramente superior na classificação do Fashion MNIST. Também demonstrou maior estabilidade nesta execução específica, já que suas métricas permaneceram equilibradas entre si, sugerindo comportamento consistente durante a avaliação. Apesar disso, as diferenças de desempenho entre logistic, tanh e relu foram pequenas, com variações modestas nas métricas finais. A principal diferença ocorreu no tempo de treinamento: logistic foi a mais lenta, enquanto relu apresentou o menor tempo, mostrando-se mais eficiente computacionalmente. Assim, logistic ofereceu o melhor resultado geral, mas com maior custo de processamento, enquanto relu manteve desempenho próximo com treinamento mais rápido.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [20]:
architectures = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

results_arch = []

for hidden_layers in architectures:
    model, metrics = run_experiment(
        X_train,
        X_val,
        y_train,
        y_val,
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        max_iter=100,
        batch_size=128,
        seed=42
    )

    results_arch.append({
        "hidden_layers": hidden_layers,
        **metrics
    })

results_arch_df = pd.DataFrame(results_arch)
results_arch_df

,hidden_layers,accuracy,precision,recall,f1_score,training_time
0,"(32,)",0.872071,0.874090,0.872071,0.872860,47.719643
1,"(64,)",0.885214,0.885065,0.885214,0.884429,69.625261
2,"(128, 64)",0.887500,0.887866,0.887500,0.887526,128.699401
3,"(256, 128)",0.896214,0.896059,0.896214,0.895783,252.204576


## Responda:

- Redes maiores sempre melhoraram os resultados?
- Redes maiores sempre melhoraram os resultados?
- Qual arquitetura apresentou melhor tradeoff?

As redes maiores apresentaram melhora gradual nos resultados neste experimento, mas esse ganho não foi proporcional ao aumento de complexidade. À medida que a arquitetura cresceu de (32,) para (256, 128), houve aumento consistente em accuracy, precision, recall e f1-score, com a arquitetura (256, 128) alcançando o melhor desempenho geral. No entanto, essa melhoria foi relativamente pequena quando comparada ao aumento expressivo no tempo de treinamento. Por exemplo, embora arquiteturas como (128, 64) e (256, 128) tenham apresentado métricas superiores, a diferença em relação à arquitetura (64,) foi modesta, enquanto o custo computacional aumentou substancialmente. Isso demonstra que redes maiores não necessariamente oferecem ganhos suficientemente relevantes para justificar sempre sua complexidade adicional. A arquitetura (64,) apresentou o melhor tradeoff neste contexto, pois manteve desempenho competitivo, relativamente próximo das arquiteturas mais complexas, mas com tempo de treinamento significativamente menor, oferecendo uma relação mais eficiente entre desempenho, simplicidade e custo computacional.

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [21]:
learning_rates = [0.1, 0.01, 0.001]

results_lr = []

for lr in learning_rates:
    model, metrics = run_experiment(
        X_train,
        X_val,
        y_train,
        y_val,
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        max_iter=100,
        batch_size=128,
        seed=42
    )

    results_lr.append({
        "learning_rate": lr,
        **metrics
    })

results_lr_df = pd.DataFrame(results_lr)
results_lr_df

,learning_rate,accuracy,precision,recall,f1_score,training_time
0,0.100,0.199857,0.148389,0.199857,0.075901,16.977441
1,0.010,0.875500,0.875834,0.875500,0.873321,123.614795
2,0.001,0.887500,0.887866,0.887500,0.887526,126.373798


## Responda:
- O treinamento ficou instável?
- Houve dificuldade de convergência?
- Qual learning rate apresentou melhor comportamento?

O treinamento apresentou instabilidade significativa com learning rate igual a 0.1. Os resultados foram drasticamente inferiores aos demais, com accuracy de aproximadamente 19,9% e f1-score muito baixo, indicando que a taxa de aprendizado foi excessivamente alta. Nesse cenário, o modelo provavelmente realizou atualizações muito grandes nos pesos, dificultando a otimização e fazendo com que o treinamento “saltasse” soluções melhores, comprometendo fortemente o aprendizado. Embora o tempo de treinamento tenha sido menor, o desempenho foi extremamente insatisfatório, mostrando que rapidez não significou qualidade.

Houve clara dificuldade de convergência com learning rate de 0.1, enquanto 0.01 e 0.001 apresentaram comportamento muito mais estável e eficaz. Ambos conseguiram convergir adequadamente, alcançando métricas elevadas e consistentes. Entre eles, o learning rate de 0.001 apresentou o melhor comportamento geral, com os maiores valores de accuracy, precision, recall e f1-score, além de maior estabilidade. Apesar de seu tempo de treinamento ter sido ligeiramente superior ao de 0.01, a diferença foi pequena frente ao ganho em desempenho. Assim, 0.001 se mostrou a escolha mais equilibrada, oferecendo melhor convergência e resultados mais confiáveis.

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


A função de ativação que apresentou melhor desempenho foi a logistic, pois obteve os maiores valores de accuracy, precision, recall e f1-score entre as ativações testadas, ainda que com tempo de treinamento superior. Entre as arquiteturas avaliadas, a (64,) apresentou o melhor tradeoff, por combinar desempenho próximo ao das redes maiores com custo computacional significativamente menor, tornando-se a opção mais equilibrada entre eficiência e qualidade preditiva. Em relação ao learning rate, o valor 0.001 demonstrou maior estabilidade, apresentando os melhores resultados gerais e convergência mais consistente, enquanto 0.1 se mostrou excessivamente alto, causando forte instabilidade e desempenho muito inferior.

Não houve evidência clara de overfitting com base nos experimentos apresentados, já que o aumento de complexidade das redes trouxe melhorias graduais sem degradação perceptível nas métricas de validação. Ainda assim, como a análise foi baseada principalmente no conjunto de validação e não houve comparação explícita entre desempenho de treino e teste final, essa conclusão deve ser interpretada com cautela.

A configuração com melhor desempenho absoluto combinou activation logistic, arquiteturas maiores como (256, 128) e learning rate 0.001, maximizando accuracy e f1-score. Entretanto, considerando custo-benefício, eficiência computacional e estabilidade, a combinação logistic + (64,) + learning rate 0.001 apresentou o melhor equilíbrio geral, mantendo desempenho competitivo com custo significativamente menor.

As principais dificuldades observadas envolveram o alto custo computacional de arquiteturas maiores, que nem sempre geraram ganhos proporcionais de desempenho, além da forte sensibilidade ao learning rate, especialmente em valores elevados, que comprometeram drasticamente a convergência. Também ficou evidente a necessidade de equilibrar desempenho e eficiência, já que configurações mais complexas frequentemente melhoraram métricas apenas de forma marginal, exigindo análise crítica para evitar aumento de custo sem benefício relevante.